# G1 Academy Bonus - Task 13: final task - perception-based delivery

## Introduction
The capstone from `notes.txt` section 11: combine perception, grasping, arm control, and SLAM navigation into one pickup -> carry -> drop-off pipeline. Every building block below was developed natively in an earlier task; this notebook assembles them one more time into a single consolidated `AcademyRobot` class - proof that, piece by piece, you could have written `sdk_wrapper.py` yourself - and then defines the end-to-end `deliver(...)` sequence:

1. `extend_arm_forward` (Task 8's `interpolate_to_ll_pose`) to a saved pre-grasp pose
2. open the hand (Task 11's `gradual_open`)
3. perception + pose estimation (Task 12's `DeliveryPipeline`)
4. IK approach in small increments (Task 10's `ik_move_ee`, via `execute_incremental_ik`)
5. close the hand at the grip pose (Task 11's `gradual_close`)
6. interpolate to a `stable_hold_pose` (Task 8's `interpolate_to_ll_pose`)
7. SLAM-navigate pickup -> dropdown while holding that pose (Task 7's `navigate_to_point`)
8. drop-off: IK to the target ArUco pose, open the hand, interpolate back to the extended pose, release ownership (Task 8's `release_arms`)

In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

_factory_config = None
def ensure_channel_factory(domain_id, interface):
    global _factory_config
    config = (int(domain_id), str(interface))
    if _factory_config is None:
        ChannelFactoryInitialize(*config)
        _factory_config = config
    elif _factory_config != config:
        raise RuntimeError(f"ChannelFactory already initialized as {_factory_config}; restart kernel for {config}.")
    return _factory_config

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 1 - `AcademyRobot`: one consolidated class over everything built in Tasks 2-12
Owns the `ChannelFactory`, the `rt/lowstate` subscriber, `rt/arm_sdk`/Dex3 publishers, `LocoClient`, `G1ArmActionClient`, the native `SlamRpc`, and a `DeliveryPipeline` instance - the same set of native components each earlier task built and exercised independently.

In [ ]:
import json
import math
import struct
from pathlib import Path

import numpy as np
import cv2
from unitree_sdk2py.g1.loco.g1_loco_client import LocoClient
from unitree_sdk2py.g1.arm.g1_arm_action_client import G1ArmActionClient
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_, unitree_hg_msg_dds__HandCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_, HandCmd_
from unitree_sdk2py.idl.std_msgs.msg.dds_ import String_
from unitree_sdk2py.rpc.client import Client
from unitree_sdk2py.utils.crc import CRC
from hand_pose_navigation.arm_fk import ArmFK
from hand_pose_navigation.arm_ik import ArmIK
from util import HAND_OPEN, HAND_CLOSED, DeliveryPipeline

WAIST_JOINTS = (12, 13, 14)
UPPER_BODY_JOINTS = list(WAIST_JOINTS) + list(range(15, 22)) + list(range(22, 29))
LEFT_ARM_JOINTS = list(range(15, 22)); RIGHT_ARM_JOINTS = list(range(22, 29))

class SlamRpc(Client):
    def __init__(self):
        super().__init__("slam_operate", False)
        for api_id in (1801, 1802, 1804, 1102, 1901):
            self._RegistApi(api_id, 0)
        self._SetApiVerson("1.0.0.1")
    def _call_json(self, api_id, payload):
        code, data = self._Call(api_id, json.dumps(payload))
        return int(code), data
    def pose_nav(self, x, y, yaw):
        qz, qw = math.sin(yaw / 2), math.cos(yaw / 2)
        return self._call_json(1102, {"data": {"targetPose": {"x": x, "y": y, "z": 0.0, "q_x": 0.0, "q_y": 0.0, "q_z": qz, "q_w": qw}, "mode": 1}})

class AcademyRobot:
    def __init__(self, iface="eth0", domain_id=0, calibration=None):
        ensure_channel_factory(domain_id, iface)
        self.loco = LocoClient(); self.loco.SetTimeout(5.0); self.loco.Init()
        self.arm_action = G1ArmActionClient(); self.arm_action.SetTimeout(10.0); self.arm_action.Init()
        self.crc = CRC()
        self.arm_sdk_pub = ChannelPublisher("rt/arm_sdk", LowCmd_); self.arm_sdk_pub.Init()
        self.hand_pubs = {"left": ChannelPublisher("rt/dex3/left/cmd", HandCmd_), "right": ChannelPublisher("rt/dex3/right/cmd", HandCmd_)}
        for pub in self.hand_pubs.values():
            pub.Init()
        self._last_hand_targets = {}
        self.slam = SlamRpc(); self.slam.SetTimeout(10.0); self.slam.Init()
        self.slam_info_sub = Latest("rt/slam_info", String_)
        self.points = json.loads(Path("slam_points.json").read_text()) if Path("slam_points.json").exists() else {}
        self.ll_poses = json.loads(Path("ll_poses.json").read_text()) if Path("ll_poses.json").exists() else {}
        self._fk, self._ik = {}, {}
        if calibration is not None:
            self.pipeline = DeliveryPipeline(*calibration)
        else:
            self.pipeline = None

    # -- state --------------------------------------------------------
    def upper_body_pose(self, timeout_s=3.0):
        deadline = time.time() + timeout_s
        while time.time() < deadline:
            if lowstate_sub.message is not None:
                return {j: float(lowstate_sub.message.motor_state[j].q) for j in UPPER_BODY_JOINTS}
            time.sleep(0.02)
        raise TimeoutError("No fresh rt/lowstate.")

    def current_pose(self):
        msg = self.slam_info_sub.message
        if msg is None:
            return None
        try:
            cur = json.loads(msg.data).get("data", {}).get("currentPose", {})
            x, y = float(cur["x"]), float(cur["y"])
            qz, qw = float(cur.get("q_z", 0.0)), float(cur.get("q_w", 1.0))
            return (x, y, math.atan2(2 * qw * qz, 1 - 2 * qz * qz))
        except Exception:
            return None

    # -- arm ------------------------------------------------------------
    def write_arm_sdk_pose(self, targets, weight=1.0, kp=30.0, kd=1.5, waist_kp=480.0, waist_kd=12.0):
        msg = unitree_hg_msg_dds__LowCmd_(); msg.mode_pr = 0; msg.mode_machine = 0
        msg.motor_cmd[29].q = max(0.0, min(1.0, float(weight)))
        for joint, q in targets.items():
            cmd = msg.motor_cmd[int(joint)]
            cmd.mode = 1; cmd.q = float(q); cmd.dq = 0.0; cmd.tau = 0.0
            cmd.kp = waist_kp if int(joint) in WAIST_JOINTS else kp
            cmd.kd = waist_kd if int(joint) in WAIST_JOINTS else kd
        msg.crc = self.crc.Crc(msg)
        self.arm_sdk_pub.Write(msg)

    def engage_arms(self, steps=50, rate_hz=50.0):
        pose = self.upper_body_pose()
        for i in range(steps + 1):
            self.write_arm_sdk_pose(pose, weight=i / steps)
            time.sleep(1.0 / rate_hz)

    def release_arms(self, steps=150, rate_hz=50.0):
        pose = self.upper_body_pose()
        for i in range(steps + 1):
            ratio = i / steps; fade = ratio * ratio * (3 - 2 * ratio); weight = 1.0 - fade
            self.write_arm_sdk_pose(pose, weight=weight, kp=30.0 * weight, kd=1.5 * weight, waist_kp=480.0 * weight, waist_kd=12.0 * weight)
            time.sleep(1.0 / rate_hz)

    def interpolate_to_ll_pose(self, name_or_pose, duration_s=4.0, steps=150):
        target = self.ll_poses[name_or_pose] if isinstance(name_or_pose, str) else name_or_pose
        target = {int(j): float(q) for j, q in target.items()}
        start = self.upper_body_pose()
        for step in range(1, steps + 1):
            ratio = step / steps; smooth = ratio * ratio * (3 - 2 * ratio)
            frame = {j: start[j] + (target[j] - start[j]) * smooth for j in target}
            self.write_arm_sdk_pose(frame)
            time.sleep(duration_s / steps)

    def extend_arm_forward(self, side="right", duration_s=4.0):
        return self.interpolate_to_ll_pose(f"extended_{side}", duration_s=duration_s)

    def fk_solver(self, side):
        if side not in self._fk:
            self._fk[side] = ArmFK(side, "urdf")
        return self._fk[side]

    def ik_solver(self, side):
        if side not in self._ik:
            self._ik[side] = ArmIK(side, "dls", max_iter=24, tol_pos_m=0.005, tol_rot_rad=0.02)
        return self._ik[side]

    def ik_move_ee(self, hand, dx=0.0, dy=0.0, dz=0.0, max_speed=0.2, max_dq=0.12, rate_hz=50.0):
        side = "right" if str(hand).lower().startswith("r") else "left"
        joints = RIGHT_ARM_JOINTS if side == "right" else LEFT_ARM_JOINTS
        current = self.upper_body_pose()
        q_init = np.array([current[j] for j in joints])
        fk = self.fk_solver(side)
        target_T = fk.compute_arm(q_init).copy()
        target_T[0, 3] += dx
        target_T[1, 3] += dy if side == "left" else -dy
        target_T[2, 3] += dz
        q_sol, info = self.ik_solver(side).solve(target_T, q_init=q_init)
        if q_sol is None:
            return {"success": False, "ik": info}
        delta = np.clip(np.asarray(q_sol) - q_init, -max_dq, max_dq)
        target_q = q_init + delta
        target = dict(current)
        for i, j in enumerate(joints):
            target[j] = float(target_q[i])
        remaining = max(abs(target[j] - current[j]) for j in joints)
        steps = max(1, int(np.ceil(remaining / max(1e-4, max_speed / rate_hz))))
        for step in range(1, steps + 1):
            ratio = step / steps; smooth = ratio * ratio * (3 - 2 * ratio)
            frame = dict(current)
            for j in joints:
                frame[j] = current[j] + (target[j] - current[j]) * smooth
            self.write_arm_sdk_pose(frame)
            time.sleep(1.0 / rate_hz)
        return {"success": True, "ik": info, "ee": tuple(float(x) for x in fk.compute_arm(target_q)[:3, 3])}

    def ik_increment(self, dx, dy, dz, side="right"):
        return self.ik_move_ee(side, dx=dx, dy=dy, dz=dz)

    def current_palm_xyz(self, side="right"):
        joints = RIGHT_ARM_JOINTS if side == "right" else LEFT_ARM_JOINTS
        current = self.upper_body_pose()
        q = np.array([current[j] for j in joints])
        return tuple(float(x) for x in self.fk_solver(side).compute_arm(q)[:3, 3])

    # -- hand -------------------------------------------------------------
    def write_hand(self, targets, side="right", kp=0.8, kd=0.05, tau=0.02):
        msg = unitree_hg_msg_dds__HandCmd_()
        for i, q in enumerate(targets):
            cmd = msg.motor_cmd[i]
            cmd.mode = (i & 0x0F) | (1 << 4); cmd.q = float(q); cmd.dq = 0.0; cmd.tau = tau; cmd.kp = kp; cmd.kd = kd
        self.hand_pubs[side].Write(msg)
        self._last_hand_targets[side] = list(targets)

    def gradual_close(self, threshold, side="right", steps=40, delay_s=0.05):
        start = self._last_hand_targets.get(side, HAND_OPEN[side])
        for step in range(1, steps + 1):
            a = step / steps
            self.write_hand([x + (y - x) * a for x, y in zip(start, HAND_CLOSED[side])], side=side)
            time.sleep(delay_s)
        return {"contact": False, "step": steps}

    def gradual_open(self, side="right", steps=40, delay_s=0.05):
        start = self._last_hand_targets.get(side, HAND_CLOSED[side])
        for step in range(1, steps + 1):
            a = step / steps
            self.write_hand([x + (y - x) * a for x, y in zip(start, HAND_OPEN[side])], side=side)
            time.sleep(delay_s)

    # -- slam ---------------------------------------------------------
    def navigate_to_point(self, point_name):
        x, y, yaw = self.points[point_name]
        return self.slam.pose_nav(x, y, yaw)

## Task 2 - `deliver(pickup_point, dropdown_point, ...)`: the end-to-end sequence
Runs the eight-step flow from `notes.txt` section 11. `stable_hold_pose` must already exist in `ll_poses.json` (save it with Task 8's `save_current_ll_pose("stable_hold_pose")` while holding a good carrying posture), and `pickup_point`/`dropdown_point` must already exist in `slam_points.json` (Task 7's `add_point`).

In [ ]:
def deliver(pickup_point, dropdown_point, side="right", marker_length_m=0.04, grasp_threshold=0.5, prompt="the delivery package"):
    robot = _robot

    # 1: pickup approach
    robot.extend_arm_forward(side=side)
    robot.gradual_open(side=side)

    # 2-3: perception + pose estimation (Task 12)
    frame = get_rgbd()
    if time.time() - frame["timestamp"] > 1.0:
        raise RuntimeError("RGB-D frame is stale.")
    rgb = cv2.imdecode(np.frombuffer(frame["rgb_jpeg"], dtype=np.uint8), cv2.IMREAD_COLOR)
    detection = robot.pipeline.detect_object(frame["rgb_jpeg"], prompt)
    if detection.confidence < 0.5:
        raise RuntimeError(f"Low-confidence detection ({detection.confidence}); aborting.")
    marker_pose = robot.pipeline.aruco_pose(rgb, marker_length_m)
    pickup_target = robot.pipeline.marker_to_palm_target(marker_pose)

    # 4: IK approach in small increments
    robot.pipeline.execute_incremental_ik(robot.ik_increment, robot.current_palm_xyz(side), pickup_target, side=side)

    # 5: close at the grip pose
    robot.gradual_close(grasp_threshold, side=side)

    # 6: interpolate to stable_hold_pose
    robot.interpolate_to_ll_pose("stable_hold_pose", duration_s=3.0)

    # 7: SLAM-navigate pickup -> dropdown while holding stable_hold_pose
    robot.navigate_to_point(dropdown_point)

    # 8: drop-off
    frame = get_rgbd()
    rgb = cv2.imdecode(np.frombuffer(frame["rgb_jpeg"], dtype=np.uint8), cv2.IMREAD_COLOR)
    dropdown_marker = robot.pipeline.aruco_pose(rgb, marker_length_m)
    dropdown_target = robot.pipeline.marker_to_palm_target(dropdown_marker)
    robot.pipeline.execute_incremental_ik(robot.ik_increment, robot.current_palm_xyz(side), dropdown_target, side=side)
    robot.gradual_open(side=side)
    robot.extend_arm_forward(side=side)
    robot.release_arms()
    return {"pickup": pickup_point, "dropdown": dropdown_point, "side": side}

# _robot = AcademyRobot(iface="eth0", domain_id=0, calibration=(camera_matrix, distortion, camera_to_base, wrist_to_palm))
# deliver("pickup", "dropdown")

## Reflect
Every method on `AcademyRobot` above is a native rebuild of something `sdk_wrapper.G1` already provides. Compare this class against `sdk_wrapper.py` end to end: same DDS init guard, same publisher/subscriber patterns, same ease-curve interpolation, same IK/SLAM/hand boundaries. Task 1 showed you the finished API; this notebook is the proof you could have built it.

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.